# 01 - Data Download and Preprocessing

This notebook prepares the image datasets used by the CycleGAN experiments.

The main experiment uses:
- Flickr photographs as the source domain (A)
- Japanese artwork as the target domain (B)

A smaller Anime dataset is also prepared so that the same pipeline can be used
for a second style-transfer experiment.

The preprocessing steps are:
1. Check that the raw datasets are available.
2. Select and preprocess the images used in the experiments.
3. Convert images to RGB and resize them to 256 × 256 pixels.
4. Create reproducible train/test splits.
5. Organize the images in the directory structure expected by CycleGAN.
6. Verify the final image counts.


In [13]:
# Imports and project paths

import os
import random
import shutil
from pathlib import Path

from PIL import Image

# Keep the dataset split reproducible.
SEED = 42
random.seed(SEED)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

TARGET_SIZE = (256, 256)
TEST_SPLIT = 0.10


In [14]:
# Check that the raw dataset folders are available

RAW_DATASETS = {
    "Flickr": RAW_DIR / "flickr" / "flickr30k_images" / "flickr30k_images",
    "WikiArt Japanese": (
        RAW_DIR
        / "wikiart"
        / "wikiart-art-movementsstyles"
        / "Japanese_Art"
        / "Japanese_Art"
    ),
    "Danbooru Anime": RAW_DIR / "danbooru" / "portraits",
}

for name, folder in RAW_DATASETS.items():
    if folder.exists():
        print(f"Found {name}: {folder}")
    else:
        print(f"Missing {name}: {folder}")

Found Flickr: ..\data\raw\flickr\flickr30k_images\flickr30k_images
Found WikiArt Japanese: ..\data\raw\wikiart\wikiart-art-movementsstyles\Japanese_Art\Japanese_Art
Found Danbooru Anime: ..\data\raw\danbooru\portraits


In [15]:
# Preprocess images from one source folder

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def preprocess_folder(
    input_dir,
    output_dir,
    target_size=TARGET_SIZE,
    max_images=None
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    # Get image files only
    image_paths = sorted(
        [
            path
            for path in input_dir.iterdir()
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )

    print(f"Found {len(image_paths)} images in {input_dir}")

    # Randomly sample if a maximum number is specified
    if max_images is not None and len(image_paths) > max_images:
        image_paths = random.sample(image_paths, max_images)
        print(f"Using random sample of {len(image_paths)} images")

    processed = 0
    skipped = 0

    for image_path in image_paths:
        output_path = output_dir / image_path.name

        # Skip images that were already processed
        if output_path.exists():
            skipped += 1
            continue

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                image = image.resize(
                    target_size,
                    Image.Resampling.LANCZOS
                )
                image.save(output_path)

            processed += 1

            if processed % 500 == 0:
                print(f"Processed {processed}/{len(image_paths)} images")

        except Exception as error:
            skipped += 1
            print(f"Skipping {image_path.name}: {error}")

    print(
        f"Finished {output_dir.name}: "
        f"{processed} processed, {skipped} skipped"
    )

In [16]:
# Verify dataset folders exist
for folder in [
    RAW_DIR / "flickr" / "flickr30k_images",
    RAW_DIR / "wikiart" / "wikiart-art-movementsstyles" / "Japanese_Art",
    RAW_DIR / "danbooru" / "portraits"
]:
    if folder.exists():
        print(f"✓ Found: {folder}")
    else:
        print(f"✗ Missing: {folder}")

✓ Found: ..\data\raw\flickr\flickr30k_images
✓ Found: ..\data\raw\wikiart\wikiart-art-movementsstyles\Japanese_Art
✓ Found: ..\data\raw\danbooru\portraits


In [17]:
# Create the train/test structure expected by CycleGAN

def create_cyclegan_dataset(
    source_a,
    source_b,
    output_root,
    test_split=TEST_SPLIT
):
    """
    Organize two image domains into the folder structure expected by CycleGAN.

    Domain A contains photographs and Domain B contains the selected
    artistic style. The two domains are split independently because
    corresponding image pairs are not required for CycleGAN training.
    """

    source_a = Path(source_a)
    source_b = Path(source_b)
    output_root = Path(output_root)

    # Recreate the CycleGAN folders so an earlier run does not leave old files behind.
    for folder in ["trainA", "trainB", "testA", "testB"]:
        folder_path = output_root / folder
        folder_path.mkdir(parents=True, exist_ok=True)

        for existing_file in folder_path.iterdir():
            if existing_file.is_file():
                existing_file.unlink()

    # Use image files only.
    images_a = sorted(
        [
            path
            for path in source_a.iterdir()
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )

    images_b = sorted(
        [
            path
            for path in source_b.iterdir()
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )

    # Shuffle each domain independently because the images are unpaired.
    random.shuffle(images_a)
    random.shuffle(images_b)

    split_a = int(len(images_a) * (1 - test_split))
    split_b = int(len(images_b) * (1 - test_split))

    train_a = images_a[:split_a]
    test_a = images_a[split_a:]

    train_b = images_b[:split_b]
    test_b = images_b[split_b:]

    # Copy the images into the CycleGAN folder structure.
    for image_path in train_a:
        shutil.copy2(image_path, output_root / "trainA" / image_path.name)

    for image_path in test_a:
        shutil.copy2(image_path, output_root / "testA" / image_path.name)

    for image_path in train_b:
        shutil.copy2(image_path, output_root / "trainB" / image_path.name)

    for image_path in test_b:
        shutil.copy2(image_path, output_root / "testB" / image_path.name)

    print(f"\nCreated dataset: {output_root.name}")
    print(f"trainA: {len(train_a)}")
    print(f"trainB: {len(train_b)}")
    print(f"testA : {len(test_a)}")
    print(f"testB : {len(test_b)}")

In [20]:
# Create the CycleGAN dataset for the main Japanese art experiment

create_cyclegan_dataset(
    PROCESSED_DIR / "temp_photos",
    PROCESSED_DIR / "temp_japanese",
    PROCESSED_DIR / "japanese_style"
)


Created dataset: japanese_style
trainA: 1800
trainB: 2011
testA : 200
testB : 224


In [21]:
create_cyclegan_dataset(
    PROCESSED_DIR / "temp_photos",
    PROCESSED_DIR / "temp_anime",
    PROCESSED_DIR / "anime_style"
)


Created dataset: anime_style
trainA: 1800
trainB: 4500
testA : 200
testB : 500


In [22]:
# Check the final CycleGAN dataset sizes

for dataset_name in ["japanese_style", "anime_style"]:
    print(f"\n{dataset_name}")

    dataset_root = PROCESSED_DIR / dataset_name

    for folder in ["trainA", "trainB", "testA", "testB"]:
        image_count = sum(
            1
            for path in (dataset_root / folder).iterdir()
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )

        print(f"{folder}: {image_count}")


japanese_style
trainA: 1800
trainB: 2011
testA: 200
testB: 224

anime_style
trainA: 1800
trainB: 4500
testA: 200
testB: 500


In [19]:
# Preprocess Flickr photos
preprocess_folder(
    RAW_DIR / "flickr" / "flickr30k_images" / "flickr30k_images",
    PROCESSED_DIR / "temp_photos",
    max_images=2000
)

preprocess_folder(
    RAW_DIR / "wikiart" /
    "wikiart-art-movementsstyles" /
    "Japanese_Art" /
    "Japanese_Art",
    PROCESSED_DIR / "temp_japanese"
)

preprocess_folder(
    RAW_DIR / "danbooru" / "portraits",
    PROCESSED_DIR / "temp_anime",
    max_images=5000
)

Found 31783 images in ..\data\raw\flickr\flickr30k_images\flickr30k_images
Using random sample of 2000 images
Processed 500/2000 images
Processed 1000/2000 images
Processed 1500/2000 images
Processed 2000/2000 images
Finished temp_photos: 2000 processed, 0 skipped
Found 2235 images in ..\data\raw\wikiart\wikiart-art-movementsstyles\Japanese_Art\Japanese_Art
Processed 500/2235 images
Processed 1000/2235 images
Processed 1500/2235 images
Processed 2000/2235 images
Finished temp_japanese: 2235 processed, 0 skipped
Found 302652 images in ..\data\raw\danbooru\portraits
Using random sample of 5000 images
Processed 500/5000 images
Processed 1000/5000 images
Processed 1500/5000 images
Processed 2000/5000 images
Processed 2500/5000 images
Processed 3000/5000 images
Processed 3500/5000 images
Processed 4000/5000 images
Processed 4500/5000 images
Processed 5000/5000 images
Finished temp_anime: 5000 processed, 0 skipped


In [ ]:
import shutil

for folder in [
    PROCESSED_DIR / "temp_photos",
    PROCESSED_DIR / "temp_japanese",
    PROCESSED_DIR / "temp_anime",
]:
    if folder.exists():
        shutil.rmtree(folder)
        print(f"Deleted {folder.name}")

print("Temporary folders removed.")

Deleted temp_photos
Deleted temp_japanese
Deleted temp_anime
Temporary folders removed.
